# Eval Mutu Jawaban dan Retrieval

Menjalankan chatbot varian lokal end-to-end di Colab, lalu menilai hasilnya dengan
dua keluarga metrik:

| Keluarga | Pustaka | Yang diukur | Konteks pembanding |
|---|---|---|---|
| Mutu jawaban akhir | RAGAS | Jawaban bersandar pada bukti, menjawab yang ditanya, cocok dengan acuan | Keluaran seluruh tool |
| Mutu retrieval | DeepEval | Relevansi, recall, dan urutan potongan yang diambil | Potongan FAQ dari vector store |

Model yang **diuji** `qwen3:1.7b`, berjalan lokal di GPU Colab. Model **juri**
`claude-haiku-4-5` lewat Claude API. Keduanya berbeda dengan sengaja: model yang menilai
jawabannya sendiri bukan pengukuran, dan juri kecil menilai terlalu berisik untuk
dipercaya.

Juri adalah satu-satunya bagian yang memakai API berbayar. Chatbot yang diukur tetap
berjalan penuh di lokal tanpa kunci API — itu justru klaim yang sedang diuji notebook ini.

## Prasyarat

1. Runtime GPU T4 — `Runtime > Change runtime type > T4 GPU`
2. Colab Secrets (ikon kunci di panel kiri) berisi:
   - `SUPABASE_URL` dan `SUPABASE_SERVICE_KEY`
   - `ANTHROPIC_API_KEY` untuk juri

Jangan pernah menulis kredensial di dalam sel. Repo ini publik.

## 1. Ambil kode

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

!git clone --branch chatbot-local-llm --depth 1 https://github.com/AryaSeptiaputra/rag-fashion-chatbot.git /content/rag-fashion-chatbot
%cd /content/rag-fashion-chatbot

## 2. Pasang dependency

Memasang chromadb, llama-index, RAGAS, dan DeepEval di atas paket bawaan Colab.
Perlu beberapa menit.

In [ ]:
!pip install -q -r requirements-eval.txt

## 3. Restart runtime

**Wajib.** Instalasi di atas mengganti numpy dan protobuf yang sudah dimuat Colab saat
runtime dinyalakan; tanpa restart, impor berikutnya gagal dengan pesan yang tidak
menunjuk penyebabnya.

Jalankan sel di bawah, tunggu runtime hidup lagi, lalu **lanjutkan dari sel 4** —
jangan mengulang dari atas.

In [ ]:
import IPython

IPython.get_ipython().kernel.do_shutdown(restart=True)

## 4. Nyalakan Ollama dan tarik model

Dijalankan setelah restart karena proses anak yang dimulai sebelum restart ikut mati.

Instalasinya dipecah jadi dua sel dengan sengaja: unduh installer, lalu jalankan.
Bentuk ringkas `curl ... | sh` menyembunyikan kegagalan, karena status keluar yang
dilaporkan adalah milik `sh`, bukan milik `curl` — installer yang gagal diunduh
membuat `sh` menerima input kosong dan keluar sukses. Akibatnya sel instalasi
terlihat berhasil dan kesalahan baru muncul beberapa sel kemudian sebagai
`FileNotFoundError: 'ollama'`, jauh dari penyebabnya.

In [ ]:
%cd /content/rag-fashion-chatbot

# Installer diunduh ke berkas lebih dulu, bukan disalurkan langsung ke sh.
# 'curl ... | sh' mengembalikan status sh, bukan status curl: kalau unduhannya
# gagal, sh menerima input kosong dan keluar dengan status 0, sehingga sel ini
# terlihat sukses padahal tidak ada yang terpasang.
!curl -fsSL https://ollama.com/install.sh -o /tmp/ollama_install.sh
!sh /tmp/ollama_install.sh

In [ ]:
import os
import shutil
import subprocess

# Installer menaruh binary di /usr/local/bin. PATH kernel Colab tidak selalu
# memuatnya sampai runtime dimulai ulang, jadi ditambahkan di sini.
os.environ["PATH"] = os.environ["PATH"] + ":/usr/local/bin"

OLLAMA = shutil.which("ollama")
if OLLAMA is None:
    raise RuntimeError(
        "Binary ollama tidak ditemukan. Jalankan ulang sel instalasi di atas dan "
        "baca keluarannya sampai habis -- kegagalan unduhan installer tidak "
        "menghentikan sel dengan sendirinya."
    )

print(subprocess.run([OLLAMA, "--version"], capture_output=True, text=True).stdout)

In [ ]:
import subprocess
import time
from pathlib import Path

import httpx

subprocess.Popen(
    [OLLAMA, "serve"],
    stdout=open("/content/ollama.log", "w"),
    stderr=subprocess.STDOUT,
)


def wait_for_ollama(timeout_seconds: int = 60) -> None:
    """Tunggu server Ollama siap menerima permintaan.

    Args:
        timeout_seconds: Batas tunggu sebelum menyerah.

    Raises:
        RuntimeError: Kalau server tidak merespons sampai batas waktu.
    """
    deadline = time.time() + timeout_seconds
    while time.time() < deadline:
        try:
            httpx.get("http://localhost:11434/api/version", timeout=2.0)
            print("Ollama siap")
            return
        except httpx.HTTPError:
            time.sleep(2)

    log = Path("/content/ollama.log").read_text(encoding="utf-8", errors="replace")
    raise RuntimeError(
        f"Ollama tidak merespons dalam {timeout_seconds} detik. "
        f"Isi /content/ollama.log:\n{log[-2000:]}"
    )


wait_for_ollama()

In [ ]:
# qwen3:1.7b       -> model yang diuji, agent sekaligus composer
# nomic-embed-text -> embedding untuk metrik RAGAS yang membutuhkannya
#
# Juri tidak ditarik ke sini: ia berjalan di Claude API, bukan di GPU Colab.
!{OLLAMA} pull qwen3:1.7b
!{OLLAMA} pull nomic-embed-text
!{OLLAMA} list

## 5. Kredensial dan konfigurasi

Env var harus diset **sebelum** `app` di-import pertama kali: `app.config.settings`
adalah singleton tingkat modul yang membaca environment sekali saat impor.

In [ ]:
import os

from google.colab import userdata

os.environ["SUPABASE_URL"] = userdata.get("SUPABASE_URL")
os.environ["SUPABASE_SERVICE_KEY"] = userdata.get("SUPABASE_SERVICE_KEY")
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

os.environ["OLLAMA_BASE_URL"] = "http://localhost:11434"
os.environ["LLM_MODEL"] = "qwen3:1.7b"
os.environ["COMPOSER_MODEL"] = "qwen3:1.7b"
os.environ["JUDGE_MODEL"] = "claude-haiku-4-5"
os.environ["JUDGE_EMBEDDING_MODEL"] = "nomic-embed-text"
os.environ["EMBEDDING_DEVICE"] = "cuda"

from app.config import settings

print("model diuji :", settings.llm_model, "/", settings.composer_model, "(lokal)")
print("model juri  :", settings.judge_model, "(Claude API)")
print("supabase    :", "terisi" if settings.supabase_url else "KOSONG")
print("kunci juri  :", "terisi" if settings.anthropic_api_key else "KOSONG")

## 6. Bangun index FAQ

Vector store Colab kosong setiap sesi, jadi FAQ di-ingest ulang dari `data/raw/faq/`.
Unduhan model embedding (~450 MB) terjadi sekali.

In [ ]:
!python scripts/ingest_faq.py

## 7. Jalankan kasus eval

Dua dataset dijalankan terpisah:

- `evals/dataset.jsonl` — 41 kasus, seluruh permukaan 8 tool. Sumber metrik **mutu jawaban**.
- `evals/retrieval.jsonl` — 16 kasus FAQ dengan acuan. Sumber metrik **mutu retrieval**.

Trace ditulis ke disk sebelum penilaian dimulai. Kalau sesi Colab putus saat menilai,
tahap ini tidak perlu diulang.

In [ ]:
!python scripts/run_eval.py \
    --dataset evals/dataset.jsonl \
    --output outputs/eval_report_answer.json \
    --trace-output outputs/trace_answer.jsonl

In [ ]:
!python scripts/run_eval.py \
    --dataset evals/retrieval.jsonl \
    --output outputs/eval_report_retrieval.json \
    --trace-output outputs/trace_retrieval.jsonl

## 8. Nilai mutu jawaban akhir (RAGAS)

Konteks pembandingnya keluaran seluruh tool, bukan potongan FAQ. `AnswerComposer`
memang hanya boleh bersandar pada hasil tool, jadi itulah bukti yang sah. Menilai
jawaban stok terhadap potongan FAQ akan menuduhnya berhalusinasi padahal datanya
benar, hanya datang dari Postgres.

In [ ]:
from pathlib import Path

from app.evals.answer import AnswerQualityEvaluator
from app.evals.judge import build_ragas_embeddings, build_ragas_judge
from app.evals.trace import load_traces

# Juri di Claude API, embedding tetap lokal lewat Ollama: Anthropic tidak
# menyediakan API embedding, dan metrik yang memakainya hanya mengukur
# kemiripan vektor, bukan memberi penilaian.
answer_report = AnswerQualityEvaluator(
    judge=build_ragas_judge(),
    embeddings=build_ragas_embeddings(),
).evaluate(load_traces(Path("outputs/trace_answer.jsonl")))

print("juri:", answer_report.judge_model)
print("agregat:", answer_report.aggregates)
print("gagal dinilai:", answer_report.unscored_total)
for note in answer_report.notes:
    print("catatan:", note)

## 9. Nilai mutu retrieval (DeepEval)

Selain tiga metrik yang dinilai LLM, `section_hit_rate` dihitung deterministik: berapa
bagian potongan yang benar-benar berasal dari seksi FAQ yang ditunjuk acuan. Metrik itu
tidak bergantung pada juri sama sekali, jadi ia tetap bisa dipercaya justru saat skor
juri terlihat mencurigakan.

In [ ]:
from app.evals.judge import build_deepeval_judge
from app.evals.retrieval import RetrievalQualityEvaluator
from app.evals.trace import FaqSectionIndex

retrieval_report = RetrievalQualityEvaluator(
    judge=build_deepeval_judge(),
    section_index=FaqSectionIndex.from_directory(),
).evaluate(load_traces(Path("outputs/trace_retrieval.jsonl")))

print("juri:", retrieval_report.judge_model)
print("agregat:", retrieval_report.aggregates)
print("gagal dinilai:", retrieval_report.unscored_total)
for note in retrieval_report.notes:
    print("catatan:", note)

## 10. Laporan gabungan

In [ ]:
import json

report_path = Path("outputs/quality_report.json")
report_path.parent.mkdir(parents=True, exist_ok=True)
report_path.write_text(
    json.dumps(
        {
            "system_under_test": {
                "agent_model": settings.llm_model,
                "composer_model": settings.composer_model,
            },
            "answer_quality": answer_report.model_dump(),
            "retrieval_quality": retrieval_report.model_dump(),
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)
print(f"Laporan lengkap disimpan di {report_path}")

In [ ]:
import pandas as pd

rows = [
    {"keluarga": report.metric_family, "metrik": metric, "skor": score}
    for report in (answer_report, retrieval_report)
    for metric, score in report.aggregates.items()
]
summary = pd.DataFrame(rows)
summary

In [ ]:
import matplotlib.pyplot as plt

figure, axis = plt.subplots(figsize=(9, 4))
axis.barh(
    [f"{row.metrik}\n({row.keluarga})" for row in summary.itertuples()],
    summary["skor"],
)
axis.set_xlim(0, 1)
axis.set_xlabel(f"skor (juri: {answer_report.judge_model})")
axis.set_title("Mutu jawaban dan retrieval")
axis.bar_label(axis.containers[0], fmt="%.2f", padding=3)
figure.tight_layout()
plt.show()

### Kasus terburuk

Rata-rata tanpa daftar kasus terburuk tidak bisa ditindaklanjuti: angka turun tanpa
petunjuk pertanyaan mana yang harus diperbaiki.

In [ ]:
for report in (answer_report, retrieval_report):
    for metric in report.aggregates:
        worst = report.worst_cases(metric, limit=3)
        print(f"\n{report.metric_family} / {metric}")
        for case in worst:
            print(f"  {case.scores[metric]:.2f}  [{case.case_id}] {case.question}")

## Cara membaca angkanya

**Juri dan yang diuji harus selalu disebut berpasangan.** Angka di sini berarti
"Qwen3-1.7B lokal, dinilai oleh `claude-haiku-4-5`". Mengganti salah satunya membuat
angkanya tidak lagi sebanding dengan run sebelumnya, jadi `judge_model` ikut tersimpan
di laporan — jangan mencatat skornya terpisah dari nama jurinya.

**Juri yang kuat memindahkan sumber keraguan, bukan menghapusnya.** Dengan Haiku 4.5
skor rendah lebih mungkin benar-benar berasal dari sistem yang diuji, bukan dari juri
yang bingung. Yang tersisa: LLM-as-judge tetap punya bias sistematis, terutama
cenderung menghukum jawaban ringkas yang sebenarnya benar.

**Korpus FAQ hanya 9 potongan dari satu dokumen contoh.** Metrik retrieval di atas 16
pertanyaan membuktikan pipeline-nya bekerja, bukan bahwa retrieval-nya bagus pada
korpus produksi. Ganti `data/raw/faq/` dengan dokumen asli sebelum menarik kesimpulan
soal kualitas.

**Gagal-nilai bukan skor nol.** `unscored_total` menghitung kasus yang jurinya gagal
mengeluarkan JSON sesuai skema. Kasus itu tidak ikut rata-rata. Angka yang besar di
sini menunjuk ke masalah transport atau rate limit, bukan ke mutu chatbot.

**Menjalankan sel penilaian memanggil API berbayar.** 57 kasus dikali beberapa metrik,
dan tiap metrik memakai lebih dari satu panggilan. Tahap generate tidak berbiaya — ia
berjalan penuh di GPU Colab — jadi iterasi prompt sebaiknya memakai lapis akurasi tool
lebih dulu, dan penilaian mutu dijalankan saat ada yang benar-benar ingin diukur.

**`section_hit_rate` tidak memakai juri.** Kalau metrik DeepEval terlihat buruk
sementara `section_hit_rate` tinggi, kecurigaan pertama harus jatuh ke jurinya.